# SYNAPSE diagnostic-fresh strength replication

Select an **A100**. Compare native, uniform and four single-candidate controls on **16 new A/M pairs**, excluding both component examples of the two previously inspected pairs. Alpha 0/0.01/0.1/1.0; states 100/500. No generator fitting or B/D/test evaluation.

[Protocol](https://github.com/akashm776/SYNAPSE/blob/main/docs/FRESH_STRENGTH_CHECK.md). Fresh means new to these diagnostics, not historically unseen by original training. Means, medians, sign counts and leave-one-out means are descriptive, not fine-tuning evidence. No winning recipe is selected.

Default: 128 groups / 1,024 one-step trials. A100 duration is unmeasured and may need multiple budget windows. The 30-minute per-invocation budget is checked between groups, not a hard timeout. Rerun after a budget pause. Do not rerun the old pilot.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
REPO = Path('/content/SYNAPSE')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/akashm776/SYNAPSE.git', str(REPO)], check=True)
CODE_REF = ''  # On resume, use the exact revision printed by your FIRST setup.
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', CODE_REF or 'origin/main'], check=True)
print('Record fresh-strength revision:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.9.1', 'torchvision==0.24.1', 'torchaudio==2.9.1', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)+'[llm,test]', 'numpy==2.3.5', 'accelerate==1.2.1'], check=True)
subprocess.run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name()); assert 'A100' in torch.cuda.get_device_name()"], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SOURCE = Path('/content/drive/MyDrive/SynLess/runs/llm_qwen_v2')
OUTPUT = Path('/content/drive/MyDrive/SYNAPSE/diagnostics/qwen_fresh_strength_v1')
CONFIG = REPO / 'configs/fresh_strength_qwen_a100.json'
assert (SOURCE / 'data.json').exists(), 'Use the full original Drive run, not the public metric archive'
print('Read-only source:', SOURCE)
print('Separate replication output:', OUTPUT)
print(CONFIG.read_text())


## Run or resume

Each state/pair/alpha group is saved. An interrupted group replays. After a runtime restart, restore the recorded CODE_REF/environment first. Do not run two sessions into one folder. Logs are streamed and appended to console.log.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'synapse.fresh_strength', '--source', str(SOURCE), '--output', str(OUTPUT), '--config', str(CONFIG)]
with (OUTPUT / 'console.log').open('a') as log:
    with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
if returncode:
    raise RuntimeError(f'Diagnostic exited {returncode}; see the traceback above or {OUTPUT / "console.log"}')
manifest = json.loads((OUTPUT / 'fresh_strength_manifest.json').read_text())
print('Fresh-strength status:', manifest['status'])
print('Excluded prior pairs:', manifest['identity']['excluded_pairs'])
if manifest['status'] == 'complete':
    print((OUTPUT / 'FRESH_STRENGTH_REPORT.md').read_text())
else:
    print('Rerun this cell to resume; the replication is not complete yet.')


## Share completed results

Share FRESH_STRENGTH_REPORT.md, fresh_strength_summary.json, fresh_strength_manifest.json, and model_buffers.json. Per-group JSON files are the resumable work; no new model checkpoints are generated. Keep original pilot and earlier diagnostic folders unchanged.
